# Lab 7/8: Camera Calibration Solution
**Goal:** Calibrate a wide-angle camera using checkerboard patterns and analyze distortion


## Cell 1: Import Libraries and Set Up

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
from pathlib import Path

# Set up paths
calib_dir = 'calibration_wide'
output_dir = calib_dir

# Checkerboard pattern size (9x6 as shown in pattern.png)
chessboard_size = (9, 6)

print(f"Calibration directory: {calib_dir}")
print(f"Checkerboard size: {chessboard_size}")
print(f"Ready to process calibration images...")

## Cell 2: Detect Checkerboard Corners in Calibration Images

In [ ]:
# Termination criteria for corner refinement
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Prepare object points (0,0,0), (1,0,0), (2,0,0)..., (8,5,0)
objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)

# Arrays to store object points and image points from all images
objpoints = []  # 3D points in real world space
imgpoints = []  # 2D points in image plane
image_sizes = []

# Get list of images
image_files = [f for f in os.listdir(calib_dir) if f.endswith('.jpg')]
image_files.sort()

print(f"Found {len(image_files)} calibration images")
print(f"Processing images...\n")

successful_detections = 0
failed_images = []

for img_file in image_files:
    img_path = os.path.join(calib_dir, img_file)
    img = cv2.imread(img_path)
    
    if img is None:
        continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    image_sizes.append(gray.shape[::-1])
    
    # Find checkerboard corners
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)
    
    if ret:
        objpoints.append(objp)
        
        # Refine corner locations
        corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners_refined)
        
        successful_detections += 1
        print(f"✓ {img_file} - Corners detected and refined")
    else:
        failed_images.append(img_file)
        print(f"✗ {img_file} - Failed to detect corners")

print(f"\nSuccessful detections: {successful_detections}/{len(image_files)}")
if failed_images:
    print(f"Failed images: {failed_images}")

## Cell 3: Calibrate Camera

In [ ]:
# Calibrate camera
img_size = image_sizes[0]

ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, img_size, None, None
)

print("Camera Calibration Results")
print("="*50)
print(f"\nReprojection Error: {ret:.4f} pixels")
print(f"\nImage Size: {img_size}")
print(f"\nCamera Matrix (K):")
print(camera_matrix)
print(f"\nDistortion Coefficients:")
print(f"  k1 (radial 1):     {dist_coeffs[0][0]:.6f}")
print(f"  k2 (radial 2):     {dist_coeffs[0][1]:.6f}")
print(f"  p1 (tangential 1): {dist_coeffs[0][2]:.6f}")
print(f"  p2 (tangential 2): {dist_coeffs[0][3]:.6f}")
print(f"  k3 (radial 3):     {dist_coeffs[0][4]:.6f}")

# Calculate field of view
focal_length = camera_matrix[0][0]
fov_x = 2 * np.arctan(img_size[0] / (2 * focal_length)) * 180 / np.pi
fov_y = 2 * np.arctan(img_size[1] / (2 * focal_length)) * 180 / np.pi

print(f"\nField of View:")
print(f"  Horizontal: {fov_x:.2f}°")
print(f"  Vertical:   {fov_y:.2f}°")

## Cell 4: Save Calibration Data

In [ ]:
# Save calibration results to pickle file
calibration_data = {
    'camera_matrix': camera_matrix,
    'dist_coeffs': dist_coeffs,
    'image_size': img_size,
    'reprojection_error': ret,
    'rvecs': rvecs,
    'tvecs': tvecs
}

output_file = os.path.join(output_dir, 'wide_dist_pickle.p')
with open(output_file, 'wb') as f:
    pickle.dump(calibration_data, f)

print(f"Calibration data saved to: {output_file}")

## Cell 5: Undistort Test Image

In [ ]:
# Load and undistort a test image
test_img_path = os.path.join(calib_dir, 'test_image.jpg')

if os.path.exists(test_img_path):
    test_img = cv2.imread(test_img_path)
    
    # Undistort the image
    h, w = test_img.shape[:2]
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dist_coeffs, (w, h), 1, (w, h)
    )
    
    undistorted = cv2.undistort(test_img, camera_matrix, dist_coeffs, None, new_camera_matrix)
    
    # Save undistorted image
    output_path = os.path.join(calib_dir, 'test_undist.jpg')
    cv2.imwrite(output_path, undistorted)
    
    print(f"Undistorted image saved to: {output_path}")
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original (Distorted)')
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Undistorted')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(calib_dir, 'distortion_comparison.png'), dpi=100)
    plt.show()
    print("\nUndistortion comparison visualization saved!")
else:
    print(f"Test image not found: {test_img_path}")

## Cell 6: Visualize Distortion on Checkerboard

In [ ]:
# Visualize distortion effect on one of the calibration images
calib_img_path = os.path.join(calib_dir, image_files[0])
calib_img = cv2.imread(calib_img_path)

if calib_img is not None:
    h, w = calib_img.shape[:2]
    new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dist_coeffs, (w, h), 1, (w, h)
    )
    
    undistorted_calib = cv2.undistort(calib_img, camera_matrix, dist_coeffs, None, new_camera_matrix)
    
    # Save undistorted calibration image
    output_path = os.path.join(calib_dir, f'undist_{image_files[0]}')
    cv2.imwrite(output_path, undistorted_calib)
    
    print(f"Undistorted calibration image saved to: {output_path}")
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(calib_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Original Distorted Checkerboard\n({image_files[0]})')
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(undistorted_calib, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Undistorted Checkerboard')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(calib_dir, 'checkerboard_distortion_comparison.png'), dpi=100)
    plt.show()
    print("\nCheckerboard distortion comparison visualization saved!")

## Cell 7: Calculate Camera Calibration Quality Metrics

In [ ]:
# Calculate per-image reprojection errors
total_error = 0
total_points = 0
per_image_errors = []

for i in range(len(objpoints)):
    # Project 3D points to image plane
    imgpoints2, _ = cv2.projectPoints(
        objpoints[i], rvecs[i], tvecs[i], camera_matrix, dist_coeffs
    )
    
    # Calculate error
    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    per_image_errors.append(error)
    
    total_error += error
    total_points += len(imgpoints2)

mean_error = total_error / len(objpoints)

print("\nCalibration Quality Analysis")
print("="*50)
print(f"Overall Reprojection Error: {ret:.4f} pixels")
print(f"Mean Error (per-image avg):  {mean_error:.4f} pixels")
print(f"Min Error:                   {min(per_image_errors):.4f} pixels")
print(f"Max Error:                   {max(per_image_errors):.4f} pixels")
print(f"Std Dev:                     {np.std(per_image_errors):.4f} pixels")
print(f"\nTotal calibration points used: {total_points}")
print(f"Number of images:             {len(objpoints)}")
print(f"\nCalibration Quality: {'EXCELLENT' if ret < 0.5 else 'GOOD' if ret < 1.0 else 'FAIR' if ret < 2.0 else 'POOR'}")

## Cell 8: Summary Report

In [ ]:
print("\n" + "="*60)
print("CAMERA CALIBRATION SUMMARY REPORT")
print("="*60)

print(f"\n📸 CAMERA SPECIFICATIONS:")
print(f"  - Type: Wide-angle camera")
print(f"  - Resolution: {img_size[0]} x {img_size[1]} pixels")
print(f"  - Focal Length (x): {camera_matrix[0][0]:.2f} pixels")
print(f"  - Focal Length (y): {camera_matrix[1][1]:.2f} pixels")
print(f"  - Principal Point: ({camera_matrix[0][2]:.2f}, {camera_matrix[1][2]:.2f})")

print(f"\n🔧 DISTORTION PARAMETERS:")
print(f"  - Radial Distortion (k1): {dist_coeffs[0][0]:.6f}")
print(f"  - Radial Distortion (k2): {dist_coeffs[0][1]:.6f}")
print(f"  - Tangential Distortion (p1): {dist_coeffs[0][2]:.6f}")
print(f"  - Tangential Distortion (p2): {dist_coeffs[0][3]:.6f}")
print(f"  - Radial Distortion (k3): {dist_coeffs[0][4]:.6f}")

print(f"\n📊 CALIBRATION QUALITY:")
print(f"  - Images Used: {successful_detections}")
print(f"  - Reprojection Error: {ret:.4f} pixels")
print(f"  - Field of View (H): {fov_x:.2f}°")
print(f"  - Field of View (V): {fov_y:.2f}°")

print(f"\n✅ OUTPUT FILES GENERATED:")
print(f"  - Calibration data: {output_file}")
print(f"  - Undistorted test image: {os.path.join(calib_dir, 'test_undist.jpg')}")
print(f"  - Undistorted checkerboard: {os.path.join(calib_dir, f'undist_{image_files[0]}')}")
print(f"  - Comparison images saved")

print("\n" + "="*60)
print("✓ Camera calibration complete!")
print("="*60)